# 1. Librairies importation

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
requirements_path = project_root / "requirements.txt"

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%pip install -r "{requirements_path}"
%pip install deepchem

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


# 2. Adapting our columns


In [2]:
import pandas as pd

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df_reg = pd.read_csv(processed_dir / "solubility_clean.csv")

# 3. Bemis-Murcko scaffolds to separate molecules with the same scaffold in training and test

In [3]:
from src.evaluation import get_scaffold


smiles_col = "Canonical_smiles"
df_reg["scaffold"] = df_reg[smiles_col].apply(get_scaffold)

invalid_count = df_reg["scaffold"].isna().sum()
print("Invalid SMILES:", invalid_count)

# Remove molecules with invalid structures.
df_reg = df_reg.dropna(subset=["scaffold"]).copy()

print("Valid molecules:", len(df_reg))
print("Unique scaffolds:", df_reg["scaffold"].nunique())


[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors


[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not r

[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors


[18:35:28] WARNING: not removing hydrogen atom without neighbors
[18:35:28] WARNING: not removing hydrogen atom without neighbors


Invalid SMILES: 0
Valid molecules: 9982
Unique scaffolds: 4996


# 4. Test-split 

In [4]:
# Size of each scaffold group
scaffold_sizes = df_reg["scaffold"].value_counts()

print(scaffold_sizes.describe())
print("Largest scaffold group:", scaffold_sizes.max())

# Splitting our molecules by scafold 
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df_reg, groups=df_reg["scaffold"])
)

train_df = df_reg.iloc[train_idx].copy()
test_df = df_reg.iloc[test_idx].copy()

print("Training set:", len(train_df), "molecules")
print("Test set:", len(test_df), "molecules")
print("Test proportion:", round(len(test_df) / len(df_reg) * 100, 2), "%")


count    4996.000000
mean        1.997998
std        25.393243
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max      1753.000000
Name: count, dtype: float64
Largest scaffold group: 1753
Training set: 8354 molecules
Test set: 1628 molecules
Test proportion: 16.31 %


# 5. Freeze the split from test-split 

In [5]:
df_reg["split"] = "train"
df_reg.loc[test_df.index, "split"] = "test"

df_reg.to_csv(processed_dir / "saved_scaffold_split_solubility.csv", index=False)

print(df_reg["split"].value_counts())
print("Split assignments saved showing which columns will be used for training and which will be used for testing.")

split
train    8354
test     1628
Name: count, dtype: int64
Split assignments saved showing which columns will be used for training and which will be used for testing.


# 6. Checking if there is any leakage between training and testing data 

In [6]:
df = pd.read_csv(project_root / "data" / "processed" / "saved_scaffold_split_solubility.csv")
train_df = df[df["split"] == "train"]
test_df = df[df["split"] == "test"]

assert set(train_df["scaffold"]).isdisjoint(set(test_df["scaffold"]))
print(train_df["Solubility"].describe())
print(test_df["Solubility"].describe())

count    8354.000000
mean       -2.896363
std         2.369664
min       -12.950000
25%        -4.313364
50%        -2.600150
75%        -1.223119
max         2.137682
Name: Solubility, dtype: float64
count    1628.000000
mean       -2.856788
std         2.360840
min       -13.171900
25%        -4.362500
50%        -2.678983
75%        -1.085275
max         1.700916
Name: Solubility, dtype: float64


# 7. Using the descriptors

In [7]:
import numpy as np
from src.features import compute_descriptors

X_train_desc = compute_descriptors(train_df["Canonical_smiles"])
X_test_desc = compute_descriptors(test_df["Canonical_smiles"])
X_train = X_train_desc.to_numpy()
X_test = X_test_desc.to_numpy()
good_columns = np.isfinite(X_train).all(axis=0) & np.isfinite(X_test).all(axis=0)
X_train = X_train[:, good_columns]
X_test = X_test[:, good_columns]

y_train = train_df["Solubility"].to_numpy()
y_test = test_df["Solubility"].to_numpy()
groups_train = train_df["scaffold"].to_numpy()

print("Descriptors computed:", X_train_desc.shape[1])
print("Train:", X_train_desc.shape, "Test:", X_test_desc.shape)


[18:35:32] WARNING: not removing hydrogen atom without neighbors
[18:35:32] WARNING: not removing hydrogen atom without neighbors
[18:35:32] WARNING: not removing hydrogen atom without neighbors


[18:35:33] WARNING: not removing hydrogen atom without neighbors
[18:35:33] WARNING: not removing hydrogen atom without neighbors
[18:35:33] WARNING: not removing hydrogen atom without neighbors
[18:35:33] WARNING: not removing hydrogen atom without neighbors
[18:35:33] WARNING: not removing hydrogen atom without neighbors
[18:35:33] WARNING: not removing hydrogen atom without neighbors


[18:35:35] WARNING: not removing hydrogen atom without neighbors
[18:35:35] WARNING: not removing hydrogen atom without neighbors
[18:35:35] WARNING: not removing hydrogen atom without neighbors


[18:35:35] WARNING: not removing hydrogen atom without neighbors
[18:35:35] WARNING: not removing hydrogen atom without neighbors
[18:35:35] WARNING: not removing hydrogen atom without neighbors


[18:35:36] WARNING: not removing hydrogen atom without neighbors
[18:35:36] WARNING: not removing hydrogen atom without neighbors
[18:35:36] WARNING: not removing hydrogen atom without neighbors
[18:35:36] WARNING: not removing hydrogen atom without neighbors
[18:35:36] WARNING: not removing hydrogen atom without neighbors
[18:35:36] WARNING: not removing hydrogen atom without neighbors


...

[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not removing hydrogen atom without neighbors
[18:36:34] WARNING: not r

[18:36:35] WARNING: not removing hydrogen atom without neighbors
[18:36:35] WARNING: not removing hydrogen atom without neighbors
[18:36:35] WARNING: not removing hydrogen atom without neighbors


[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors


[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors
[18:36:36] WARNING: not removing hydrogen atom without neighbors


Descriptors computed: 210
Train: (8354, 210) Test: (1628, 210)


# 7.1 : Saving results from #7 to avoird running it everytime (6min)

In [8]:
import numpy as np

X_train_all = X_train_desc.to_numpy()
X_test_all = X_test_desc.to_numpy()
np.save(project_root / "data" / "processed" / "Solubility_X_train_all.npy", X_train_all)
np.save(project_root / "data" / "processed" / "Solubility_X_test_all.npy", X_test_all)


# 8. Cross Valditation (on training data only) 

In [9]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler

# remove the descriptors that have the same value for all molecules
selector = VarianceThreshold(threshold=0.0)
X_train_s = selector.fit_transform(X_train)
X_test_s = selector.transform(X_test)

# normalize: the scaler learns only from the training set
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train_s)
X_test_s = scaler.transform(X_test_s)

print("Train:", X_train_s.shape, "Test:", X_test_s.shape)

Train: (8354, 195) Test: (1628, 195)


/Users/reginaseca/miniforge3/envs/psl/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1746: RuntimeWarning: overflow encountered in multiply
  sqr = np.multiply(arr, arr, out=arr, where=where)


In [10]:
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, 20, 30]
}

# 5-folds cross validation (we keep the molecules with the same scaffold together)
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=0),
    param_grid,
    cv=GroupKFold(n_splits=5),
    scoring="neg_root_mean_squared_error"
)

# only training data 
grid_search.fit(X_train_s, y_train, groups=groups_train)

print("Best parameters:", grid_search.best_params_)
print("Best RMSE (validation):", round(-grid_search.best_score_, 3))

Best parameters: {'max_depth': 30, 'n_estimators': 100}
Best RMSE (validation): 1.118


# 8.1 To avoid running [13] again (around 18 min)

In [11]:
import json
import pandas as pd

processed_dir = project_root / "data" / "processed"

# 1. save the best parameters
with open(processed_dir / "rf_best_params.json", "w") as f:
    json.dump(grid_search.best_params_, f)

# 2. save the results of all the combinations tested
pd.DataFrame(grid_search.cv_results_).to_csv(
    processed_dir / "rf_grid_search_results.csv", index=False
)

print("Saved in:", processed_dir)
print("Best parameters:", grid_search.best_params_)
print("Best RMSE (validation):", round(-grid_search.best_score_, 3))

Saved in: /Users/reginaseca/Documents/M2_AIH/Courses/Cheminformatics/Data-Project-for-Cheminformatics-class/data/processed
Best parameters: {'max_depth': 30, 'n_estimators': 100}
Best RMSE (validation): 1.118


# 9. Train and evaluate the model 
9.1 Reload the saved data and the best hyperparameters

In [12]:
import json
import numpy as np
import pandas as pd


from pathlib import Path
cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "data" / "processed").exists()), cwd)
processed_dir = project_root / "data" / "processed"

if "X_train_s" in globals() and "groups_train" in globals():
    print("X_train_s, X_test_s, y_train, y_test and groups_train are already defined.")
else:
    from sklearn.feature_selection import VarianceThreshold
    from sklearn.preprocessing import MinMaxScaler

    df = pd.read_csv(processed_dir / "saved_scaffold_split_solubility.csv")
    train_df = df[df["split"] == "train"]
    test_df = df[df["split"] == "test"]

    y_train = train_df["Solubility"].values
    y_test = test_df["Solubility"].values
    groups_train = train_df["scaffold"].values

    X_train_all = np.load(processed_dir / "Solubility_X_train_all.npy")
    X_test_all = np.load(processed_dir / "Solubility_X_test_all.npy")

    # same preprocessing as in sections 7 and 8
    good_columns = np.isfinite(X_train_all).all(axis=0) & np.isfinite(X_test_all).all(axis=0)
    X_train = X_train_all[:, good_columns]
    X_test = X_test_all[:, good_columns]

    selector = VarianceThreshold(threshold=0.0)
    X_train_s = selector.fit_transform(X_train)
    X_test_s = selector.transform(X_test)

    scaler = MinMaxScaler()
    X_train_s = scaler.fit_transform(X_train_s)
    X_test_s = scaler.transform(X_test_s)

print("Train:", X_train_s.shape, "Test:", X_test_s.shape)

# the best hyperparameters found by the grid search (saved in section 8.1)
with open(processed_dir / "rf_best_params.json") as f:
    best_params = json.load(f)
print("Best parameters:", best_params)

# the results of all the combinations tested (RMSE = - neg_root_mean_squared_error)
grid_results = pd.read_csv(processed_dir / "rf_grid_search_results.csv")
grid_results["RMSE validation"] = -grid_results["mean_test_score"]
grid_results[["param_max_depth", "param_n_estimators", "RMSE validation",
              "std_test_score", "rank_test_score"]].sort_values("rank_test_score").round(3)

X_train_s, X_test_s, y_train, y_test and groups_train are already defined.
Train: (8354, 195) Test: (1628, 195)
Best parameters: {'max_depth': 30, 'n_estimators': 100}


,param_max_depth,param_n_estimators,RMSE validation,std_test_score,rank_test_score
7,30,100,1.118,0.121,1
5,20,100,1.118,0.119,2
6,30,50,1.127,0.122,3
4,20,50,1.127,0.118,4
3,10,100,1.143,0.125,5
2,10,50,1.149,0.125,6
1,5,100,1.256,0.158,7
0,5,50,1.260,0.158,8


Conclusion : The best model is the one with max_depth=30 and n_estimators=100, with a validation RMSE of 1.107. 

The depth changes the results the most: with max_depth=5 the RMSE is around 1.25, and it gets better until depth 20. After that it doesn’t really change (1.108 for depth 20 and 1.107 for depth 30). 

The number of trees matters very little, 100 trees is just a bit better than 50.

The std between the folds is about 0.115, so it is bigger than the differences between the best models. So I think depth 20 and depth 30 are basically the same, but we keep the best one from the grid search. We will see in section 11 if the test RMSE is close to 1.107.


9.2 The function train_test_model trains a model on the training set, predicts the train and test sets, prints the RMSE of both and returns them, so that all the models are evaluated in the same way.

In [13]:
from sklearn.metrics import mean_squared_error

def train_test_model(model, X_train, y_train, X_test, y_test):
    """
    Function that trains a model, and tests it.
    Inputs: sklearn model, train_data, test_data
    """
    # Train model
    model.fit(X_train, y_train)

    # Calculate RMSE on training and test
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    model_train_mse = mean_squared_error(y_train, y_pred_train)
    model_test_mse = mean_squared_error(y_test, y_pred_test)
    model_train_rmse = model_train_mse ** 0.5
    model_test_rmse = model_test_mse ** 0.5
    print(f"RMSE on train set: {model_train_rmse:.3f}, and test set: {model_test_rmse:.3f}.\n")

    return model_train_rmse, model_test_rmse

# 10. Create models and finding the best one 
A baseline that always predicts the mean of the training set gives the minimum RMSE to beat.

In [14]:
# baseline: always predicts the mean of the training targets
from sklearn.dummy import DummyRegressor
dummy_reg = DummyRegressor(strategy="mean")

# random forest regressor, and the default criterion is mean squared error (MSE)
from sklearn.ensemble import RandomForestRegressor
ranf_reg = RandomForestRegressor(n_estimators=10, random_state=0)  # using 10 trees and seed=0

# XGBoost regressor
from xgboost import XGBRegressor
xgb_reg = XGBRegressor(n_estimators=10, random_state=0)  # using 10 trees and seed=0


In [15]:
results = {}

# Train and test the baseline
print("Evaluating Baseline (mean).")
results["Baseline (mean)"] = train_test_model(dummy_reg, X_train_s, y_train, X_test_s, y_test)

# Train and test the random forest model
print("Evaluating Random Forest Model.")
results["Random Forest (default)"] = train_test_model(ranf_reg, X_train_s, y_train, X_test_s, y_test)

# Train and test XGBoost model
print("Evaluating XGBoost model.")
results["XGBoost (default)"] = train_test_model(xgb_reg, X_train_s, y_train, X_test_s, y_test)


Evaluating Baseline (mean).
RMSE on train set: 2.370, and test set: 2.360.

Evaluating Random Forest Model.


RMSE on train set: 0.457, and test set: 1.182.

Evaluating XGBoost model.


RMSE on train set: 0.875, and test set: 1.154.



# 10.1 Which model is better in this case ?

All the models are much better than the baseline (2.360), so they really learned something. On the test set, XGBoost is the best (1.174), and then the Random Forest (1.196). But the difference between XGBoost and the Random Forest is small (0.02), so I would not say that one is clearly better, especially because the RF only has 10 trees here.

The RF has a very low train RMSE (0.458) and a much higher test RMSE (1.196), so it overfits a lot. XGBoost overfits less (0.889 vs 1.174). 

For now XGBoost is the best, but all these models use default parameters. 

So we will see in section 11 if the Random Forest gets better with the best hyperparameters.

# 11. Hyperparameter optimization
The best hyperparameters were found with GridSearchCV on the training data only (grid_search.fit(X_train_s, y_train, groups=groups_train)), using GroupKFold so that molecules with the same scaffold stay together. The test set was not used. We now re-train the random forest with these parameters and evaluate it on the test set.

In [16]:
# re-train a model using best hyperparameters
rf_gs = RandomForestRegressor(**best_params, random_state=0)

print('Random forests performance after hyperparameter optimization:')
results["Random Forest (GridSearch)"] = train_test_model(rf_gs, X_train_s, y_train, X_test_s, y_test)

Random forests performance after hyperparameter optimization:


RMSE on train set: 0.389, and test set: 1.103.



**Results analysis**

The test RMSE is 1.105, much better than the baseline (2.360). 

The train RMSE is lower (0.389), so there is still some overfitting. 

But the test RMSE is almost the same as the validation RMSE from the grid search (1.107), so we can say the model generalizes well to new scaffolds.

**Comparison of `ranf_reg` and `rf_gs`**

Both are Random Forests. `ranf_reg` is the default one from section 10 (10 trees, no hyperparameter tuning), and `rf_gs` is the one with the best hyperparameters from the grid search (`max_depth=30`, `n_estimators=100`).

`rf_gs` is better: its test RMSE is 1.105 and the default RF has 1.196, and both are way better than the baseline (2.360). But both overfit a lot, because the train RMSE is much lower than the test one (0.389 vs 1.105 for `rf_gs`). So the optimization didn't really fix the overfitting, it just made the test error smaller.

The test RMSE (1.105) is almost the same as the validation RMSE from the grid search (1.107), so I think the model generalizes well to new scaffolds.

# 12. Summary of the results and written conclusion 

In [17]:
summary = pd.DataFrame(results, index=["RMSE train", "RMSE test"]).T
summary["Gap (test - train)"] = summary["RMSE test"] - summary["RMSE train"]
summary.sort_values("RMSE test").round(3)

,RMSE train,RMSE test,Gap (test - train)
Random Forest (GridSearch),0.389,1.103,0.714
XGBoost (default),0.875,1.154,0.279
Random Forest (default),0.457,1.182,0.725
Baseline (mean),2.370,2.360,-0.009


# 13. Compare representations and save regression results

This section uses the shared feature functions from `src/` to compare RDKit descriptors and Morgan fingerprints with Random Forest and XGBoost regressors. The scaffold-aware training folds are used for model selection, and the held-out test set is evaluated only at the end.


In [18]:
import numpy as np
from src.features import compute_descriptors, compute_morgan

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

# Calculate both representations from the shared src helpers.
X_train_desc = compute_descriptors(train_df["Canonical_smiles"])
X_test_desc = compute_descriptors(test_df["Canonical_smiles"])
X_train_morgan = compute_morgan(train_df["Canonical_smiles"])
X_test_morgan = compute_morgan(test_df["Canonical_smiles"])

# Keep invalid or non-representable descriptor values as NaN for imputation.
descriptor_limit = np.finfo(np.float32).max
X_train_desc = X_train_desc.mask(
    ~np.isfinite(X_train_desc) | (X_train_desc.abs() > descriptor_limit)
)
X_test_desc = X_test_desc.mask(
    ~np.isfinite(X_test_desc) | (X_test_desc.abs() > descriptor_limit)
)

features = {
    "descriptors": (X_train_desc, X_test_desc),
    "Morgan": (X_train_morgan, X_test_morgan),
}

print({name: (train.shape, test.shape) for name, (train, test) in features.items()})


[18:46:38] WARNING: not removing hydrogen atom without neighbors
[18:46:38] WARNING: not removing hydrogen atom without neighbors
[18:46:38] WARNING: not removing hydrogen atom without neighbors


[18:46:39] WARNING: not removing hydrogen atom without neighbors
[18:46:39] WARNING: not removing hydrogen atom without neighbors
[18:46:39] WARNING: not removing hydrogen atom without neighbors
[18:46:39] WARNING: not removing hydrogen atom without neighbors
[18:46:39] WARNING: not removing hydrogen atom without neighbors
[18:46:39] WARNING: not removing hydrogen atom without neighbors


[18:46:41] WARNING: not removing hydrogen atom without neighbors
[18:46:41] WARNING: not removing hydrogen atom without neighbors
[18:46:41] WARNING: not removing hydrogen atom without neighbors


[18:46:41] WARNING: not removing hydrogen atom without neighbors
[18:46:41] WARNING: not removing hydrogen atom without neighbors
[18:46:41] WARNING: not removing hydrogen atom without neighbors


[18:46:42] WARNING: not removing hydrogen atom without neighbors
[18:46:42] WARNING: not removing hydrogen atom without neighbors
[18:46:42] WARNING: not removing hydrogen atom without neighbors
[18:46:42] WARNING: not removing hydrogen atom without neighbors
[18:46:42] WARNING: not removing hydrogen atom without neighbors
[18:46:42] WARNING: not removing hydrogen atom without neighbors


...

[18:47:49] WARNING: not removing hydrogen atom without neighbors
[18:47:49] WARNING: not removing hydrogen atom without neighbors
[18:47:49] WARNING: not removing hydrogen atom without neighbors
[18:47:49] WARNING: not removing hydrogen atom without neighbors
[18:47:49] WARNING: not removing hydrogen atom without neighbors
[18:47:49] WARNING: not removing hydrogen atom without neighbors
[18:47:49] WARNING: not removing hydrogen atom without neighbors
[18:47:49] WARNING: not removing hydrogen atom without neighbors
[18:47:49] WARNING: not removing hydrogen atom without neighbors
[18:47:49] WARNING: not removing hydrogen atom without neighbors


[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not r

[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors
[18:47:55] WARNING: not removing hydrogen atom without neighbors


[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors
[18:48:51] WARNING: not removing hydrogen atom without neighbors


{'descriptors': ((8354, 210), (1628, 210)), 'Morgan': ((8354, 2048), (1628, 2048))}


In [19]:
# Keep imputation inside each model so preprocessing is learned from training folds only.
models = {
    "Random Forest": (
        RandomForestRegressor(random_state=0, n_jobs=-1),
        {
            "model__n_estimators": [100],
            "model__max_depth": [10, 30],
        },
    ),
    "XGBoost": (
        XGBRegressor(
            objective="reg:squarederror",
            random_state=0,
            n_jobs=1,
            verbosity=0,
        ),
        {
            "model__n_estimators": [100],
            "model__max_depth": [3, 6],
            "model__learning_rate": [0.05],
        },
    ),
}

cv = GroupKFold(n_splits=5)
scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
}

searches = {}
default_rows = []
tuned_rows = []

for representation, (X_train, X_test) in features.items():
    for model_name, (estimator, parameter_grid) in models.items():
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", estimator),
        ])
        key = f"{model_name} + {representation}"

        default_cv = GridSearchCV(
            pipeline, {"model": [estimator]}, cv=cv, scoring=scoring,
            refit="RMSE", return_train_score=False, error_score="raise",
        )
        default_cv.fit(X_train, y_train, groups=groups_train)
        default_result = {"Model": key}
        for metric in scoring:
            default_result[f"{metric} mean"] = default_cv.cv_results_[f"mean_test_{metric}"][0]
            default_result[f"{metric} std"] = default_cv.cv_results_[f"std_test_{metric}"][0]
        default_rows.append(default_result)

        search = GridSearchCV(
            pipeline, parameter_grid, cv=cv, scoring=scoring, refit="RMSE",
            return_train_score=False, error_score="raise",
        )
        search.fit(X_train, y_train, groups=groups_train)
        searches[key] = (search, X_test)
        tuned_result = {"Model": key, "Best parameters": str(search.best_params_)}
        for metric in scoring:
            tuned_result[f"{metric} mean"] = search.cv_results_[f"mean_test_{metric}"][search.best_index_]
            tuned_result[f"{metric} std"] = search.cv_results_[f"std_test_{metric}"][search.best_index_]
        tuned_rows.append(tuned_result)

default_cv_results = pd.DataFrame(default_rows)
tuned_cv_results = pd.DataFrame(tuned_rows).sort_values("RMSE mean", ascending=False)

display(default_cv_results.round(3))
display(tuned_cv_results.round(3))


,Model,RMSE mean,RMSE std,MAE mean,MAE std,R2 mean,R2 std
0,Random Forest + descriptors,-1.108,0.115,-0.776,0.064,0.763,0.052
1,XGBoost + descriptors,-1.090,0.113,-0.761,0.066,0.771,0.048
2,Random Forest + Morgan,-1.750,0.195,-1.352,0.243,0.280,0.575
3,XGBoost + Morgan,-1.656,0.098,-1.284,0.127,0.393,0.394


,Model,Best parameters,RMSE mean,RMSE std,MAE mean,MAE std,R2 mean,R2 std
1,XGBoost + descriptors,"{'model__learning_rate': 0.05, 'model__max_dep...",-1.085,0.112,-0.772,0.068,0.773,0.049
0,Random Forest + descriptors,"{'model__max_depth': 30, 'model__n_estimators'...",-1.109,0.117,-0.777,0.064,0.763,0.050
3,XGBoost + Morgan,"{'model__learning_rate': 0.05, 'model__max_dep...",-1.767,0.075,-1.398,0.102,0.325,0.395
2,Random Forest + Morgan,"{'model__max_depth': 30, 'model__n_estimators'...",-1.785,0.183,-1.389,0.229,0.259,0.573


In [20]:
# Select the lowest validation RMSE and evaluate that model once on the untouched test set.
best_model_name = tuned_cv_results.sort_values("RMSE mean", ascending=False).iloc[0]["Model"]
selected_search, X_selected_test = searches[best_model_name]
final_model = selected_search.best_estimator_
y_test_pred = final_model.predict(X_selected_test)

final_test_results = pd.DataFrame([{
    "Model": best_model_name,
    "RMSE": mean_squared_error(y_test, y_test_pred) ** 0.5,
    "MAE": mean_absolute_error(y_test, y_test_pred),
    "R2": r2_score(y_test, y_test_pred),
}])

test_predictions = test_df[["Drug_ID", "Canonical_smiles", "Solubility"]].copy()
test_predictions["predicted_solubility"] = y_test_pred
test_predictions["residual"] = test_predictions["Solubility"] - test_predictions["predicted_solubility"]
test_predictions["model"] = best_model_name

print("Selected model:", best_model_name)
display(final_test_results.round(3))
display(test_predictions.head())


Selected model: XGBoost + descriptors


,Model,RMSE,MAE,R2
0,XGBoost + descriptors,1.082,0.762,0.79


,Drug_ID,Canonical_smiles,Solubility,predicted_solubility,residual,model
12,"2,3-dimethylphenol; 2,4-dimethylphenol; 2,5-di...",Cc1cc(C)cc(O)c1.Cc1ccc(C)c(O)c1.Cc1ccc(O)c(C)c...,-1.980310,-5.351586,3.371277,XGBoost + descriptors
17,zinc bis[O-(2-ethylhexyl) O-isobutyl dithiopho...,CCCCC(CC)COP(=S)([S-])OCC(C)C.CCCCC(CC)COP(=S)...,-2.958195,-4.065950,1.107755,XGBoost + descriptors
20,2-methylheptyl 3-sulfanylpropanoate,CC(C)CCCCCOC(=O)CCS,-4.454382,-3.705297,-0.749085,XGBoost + descriptors
27,"(1S,5S)-6,6-dimethyl-2-methylidenebicyclo[3.1....",C=C1CCC2CC1C2(C)C,-4.292313,-3.920396,-0.371917,XGBoost + descriptors
38,molybdenum,[Mo],-4.203848,-5.868065,1.664216,XGBoost + descriptors


In [21]:
# Save files in the same style as the hERG workflow.
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

default_cv_results.to_csv(results_dir / "solubility_default_cv_results.csv", index=False)
tuned_cv_results.to_csv(results_dir / "solubility_tuned_cv_results.csv", index=False)
final_test_results.to_csv(results_dir / "solubility_test_results.csv", index=False)
test_predictions.to_csv(results_dir / "solubility_test_predictions.csv", index=False)

best_params = {
    name: {key.replace("model__", ""): value for key, value in search.best_params_.items()}
    for name, (search, _) in searches.items()
}
with (results_dir / "solubility_best_params.json").open("w") as handle:
    json.dump(best_params, handle, indent=2)

print("Saved regression results in:", results_dir)


Saved regression results in: /Users/reginaseca/Documents/M2_AIH/Courses/Cheminformatics/Data-Project-for-Cheminformatics-class/results
